# 01 - 資料驗證

## 目的

本筆記本旨在驗證專案中使用的公開製造業資料集，然後再進行任何預測或規劃分析。

本項目專注於美國電子元件製造業的需求和訂單履行規劃。

### 主要資料系列

- **新訂單**：新增製造需求

- **出貨量**：已完成並已出貨的製造價值

- **未完成訂單**：未完成的訂單積壓

- **總庫存**：製造庫存

- **工業生產指數**：半導體和電子元件的生產活動

## 資料來源

- 美國人口普查局 - 製造商出貨量、庫存和訂單 (M3)

- 聯準會 - 工業生產和產能利用率

## 驗證目標

本筆記本檢查：

1. 是否可以成功下載所有必要的資料系列。

2. 日期和數值是否有效。

3. 是否存在重複的月份。

4. 是否存在月度觀測資料缺失的情況。

5. 是否可以將資料集合併成一個月度規劃資料集。

## Step 1｜載入分析套件

### 這一格要做什麼？

這一格先把後面資料驗證會使用的 Python 套件載入。

- `Path`：處理專案資料夾與檔案路徑。
- `numpy`：處理數值資料。
- `pandas`：讀取、整理與驗證表格資料。
- `matplotlib.pyplot`：後續需要畫圖時使用。
- `pd.set_option(...)`：讓 DataFrame 在 Notebook 裡顯示得更完整。

### 為什麼要先做？

如果套件沒有成功載入，後面的資料下載、合併、缺值檢查都無法執行。因此第一步先確認執行環境正常。


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries loaded successfully.")

Libraries loaded successfully.


### 輸出解讀

輸出：

`Libraries loaded successfully.`

代表這一格使用到的 Python 套件都成功載入，沒有 `ModuleNotFoundError` 或 Import Error。

**白話來說：**目前 Python 環境是正常的，可以繼續進行資料讀取與驗證。

**下一步：**設定專案資料夾，讓 Raw Data、Processed Data 和報告都有固定位置。


## Step 2｜設定專案資料夾路徑

### 這一格要做什麼？

這一格定義整個專案會使用的資料夾位置：

- `data/raw`：保存從官方來源下載、尚未加工的原始資料。
- `data/processed`：保存已經驗證、整理完成的分析資料。
- `reports/tables`：保存驗證結果與之後的分析表格。
- `reports/figures`：保存之後產生的圖表。

程式也會使用 `mkdir(..., exist_ok=True)` 自動建立缺少的資料夾。

### 為什麼要這樣分？

把 Raw Data、Processed Data 和分析結果分開，可以避免之後不小心修改原始資料，也能讓 GitHub 專案結構更清楚、容易重現。


In [2]:
#設定專案路徑
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    REPORT_TABLE_DIR,
    REPORT_FIGURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)

Project root: C:\Users\momo8\Desktop\electronic-components-planning
Raw data directory: C:\Users\momo8\Desktop\electronic-components-planning\data\raw


### 輸出解讀

目前輸出顯示：

- Project root：`C:\Users\momo8\Desktop\electronic-components-planning`
- Raw data directory：`...\electronic-components-planning\data\raw`

這與目前的專案結構一致。

**白話來說：**Notebook 確實把 `electronic-components-planning` 當成專案根目錄，後面下載的原始資料會進到正確的 `data/raw`，沒有存到 Desktop 或其他錯誤位置。

**結論：**路徑設定正常，可以繼續。


## Step 3｜定義要使用的 5 個官方資料系列

### 這一格要做什麼？

這一格不是開始分析，而是先建立一張「資料字典」，明確記錄每一個資料系列的：

- FRED `series_id`
- 在本專案中的名稱
- 原始資料來源
- 單位

本專案使用 4 個 U.S. Census Bureau M3 製造業系列，加上 1 個 Federal Reserve 工業生產指數。

### 為什麼這一步重要？

如果後面只看到 `A34HNO`、`A34HVS` 這些代碼，很難知道它們代表什麼。先集中定義後，下載、驗證與分析時都能使用同一套名稱，也比較不容易抓錯資料。


In [3]:
#定義到底要抓哪些資料
SERIES_CONFIG = {
    "new_orders": {
        "series_id": "A34HNO",
        "name": "New Orders",
        "source": "U.S. Census Bureau M3",
        "unit": "Millions of Dollars",
    },
    "shipments": {
        "series_id": "A34HVS",
        "name": "Shipments",
        "source": "U.S. Census Bureau M3",
        "unit": "Millions of Dollars",
    },
    "unfilled_orders": {
        "series_id": "A34HUO",
        "name": "Unfilled Orders",
        "source": "U.S. Census Bureau M3",
        "unit": "Millions of Dollars",
    },
    "inventories": {
        "series_id": "A34HTI",
        "name": "Total Inventories",
        "source": "U.S. Census Bureau M3",
        "unit": "Millions of Dollars",
    },
    "industrial_production": {
        "series_id": "IPG3344S",
        "name": "Industrial Production",
        "source": "Federal Reserve Board",
        "unit": "Index 2017=100",
    },
}

pd.DataFrame(SERIES_CONFIG).T

,series_id,name,source,unit
new_orders,A34HNO,New Orders,U.S. Census Bureau M3,Millions of Dollars
shipments,A34HVS,Shipments,U.S. Census Bureau M3,Millions of Dollars
unfilled_orders,A34HUO,Unfilled Orders,U.S. Census Bureau M3,Millions of Dollars
inventories,A34HTI,Total Inventories,U.S. Census Bureau M3,Millions of Dollars
industrial_production,IPG3344S,Industrial Production,Federal Reserve Board,Index 2017=100


### 輸出解讀

這張表確認本專案要使用的五個資料系列都已經定義完成：

| 專案欄位 | Series ID | 白話意思 |
|---|---|---|
| `new_orders` | A34HNO | 本月新進來的製造訂單需求 |
| `shipments` | A34HVS | 本月已完成並出貨的製造價值 |
| `unfilled_orders` | A34HUO | 已接單但月底仍未完成的 backlog |
| `inventories` | A34HTI | 月底製造庫存 |
| `industrial_production` | IPG3344S | 半導體與其他電子元件實質產出活動指數 |

前四個 M3 指標的單位都是 **Millions of Dollars**，因此之後可以在同一業務脈絡下比較 Orders、Shipments、Backlog 與 Inventory。

`industrial_production` 則是 **Index 2017=100**，它不是金額，因此**不能直接拿 0.36、100、190 和 5,000 百萬美元比較大小**；它主要用來觀察生產活動的相對變化。

**下一步：**建立統一下載函數，讓五份資料用相同方法抓取與整理。


## Step 4｜建立 FRED 資料下載函數

### 這一格要做什麼？

建立 `download_fred_series()` 函數，讓後面只要給它：

1. FRED Series ID
2. 想使用的欄位名稱

就能自動：

- 從 FRED CSV 端點下載資料。
- 把欄位統一成 `date + 指標名稱`。
- 將日期轉成 datetime。
- 將數值欄位轉成數字格式。

### 為什麼不一份一份手動下載？

五個 Series 如果分別寫五套讀檔程式，很容易格式不一致。做成函數後，每份資料都經過完全相同的整理步驟，後面也更容易重現。


In [4]:
#建立下載函數
def download_fred_series(series_id: str, column_name: str) -> pd.DataFrame:
    """
    Download one public FRED series as a two-column DataFrame.

    Parameters
    ----------
    series_id:
        FRED series identifier.

    column_name:
        Name used for the value column in this project.
    """

    url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={series_id}"
    )

    dataframe = pd.read_csv(url)

    dataframe.columns = ["date", column_name]

    dataframe["date"] = pd.to_datetime(
        dataframe["date"],
        errors="coerce",
    )

    dataframe[column_name] = pd.to_numeric(
        dataframe[column_name],
        errors="coerce",
    )

    return dataframe

### 輸出解讀

這一格沒有畫面輸出是正常的，因為这里只是「定義函數」，還沒有真正呼叫它。

**白話來說：**我們現在只是先做出一個下載工具。Python 已經知道之後呼叫 `download_fred_series()` 時應該做哪些事情。

**下一步：**先用 New Orders 實際測試一次，確認這個工具真的可以正常工作。


## Step 5｜先測試一個 Series：New Orders

### 這一格要做什麼？

正式下載五份資料之前，先只測試 `A34HNO`（New Orders）。

這裡先看：

- 前 5 筆資料 `head()`
- 最後 5 筆資料 `tail()`

### 為什麼不要一開始五份全部抓？

如果下載函數或欄位轉換有錯，只測一份比較容易定位問題。先確認一份資料長得正確，再批次下載其他 Series。


In [5]:
#先只測一份
new_orders_test = download_fred_series(
    series_id="A34HNO",
    column_name="new_orders",
)

print(new_orders_test.head())
print()
print(new_orders_test.tail())

        date  new_orders
0 1992-02-01        3411
1 1992-03-01        3711
2 1992-04-01        3745
3 1992-05-01        3735
4 1992-06-01        3814

          date  new_orders
410 2026-04-01        5405
411 2026-05-01        5579
412 2026-06-01        5668
413 2026-07-01        5711
414 2026-08-01        5769


### 輸出解讀

New Orders 成功下載，而且畫面顯示：

- 最前面的資料從 **1992-02-01** 開始。
- Notebook 執行時最新資料到 **2026-08-01**。
- `date` 與 `new_orders` 兩欄都正確出現。

例如：

- 1992-02：3,411 百萬美元
- 2026-08：5,769 百萬美元

**白話來說：**FRED 資料已經成功抓進 Python，下載函數可以正常使用。

另外要注意：New Orders 官方系列本身就是從 **1992-02** 才開始，不是我們漏掉 1992-01。

**下一步：**不能只看頭尾正常，還要檢查資料型態、空值與完整日期範圍。


## Step 6｜檢查 New Orders 的基本資料型態

### 這一格要做什麼？

這一步確認剛才下載的 New Orders 是否符合最基本的資料品質要求：

- `Shape`：總共有幾列、幾欄。
- `dtypes`：日期是不是日期格式、訂單是不是數值格式。
- `Missing values`：有沒有空值。
- `Date range`：資料從哪一個月開始，到哪一個月結束。

### 為什麼這一步不能省？

畫面上看起來像日期，不代表 Python 真的把它當日期；看起來像數字，也可能其實是文字。先確認型態，後面的時間序列分析才不會出錯。


In [6]:
#確認下載資料基本型態
print("Shape:", new_orders_test.shape)

print("\nData types:")
print(new_orders_test.dtypes)

print("\nMissing values:")
print(new_orders_test.isna().sum())

print("\nDate range:")
print("Start:", new_orders_test["date"].min())
print("End:", new_orders_test["date"].max())

Shape: (415, 2)

Data types:
date          datetime64[us]
new_orders             int64
dtype: object

Missing values:
date          0
new_orders    0
dtype: int64

Date range:
Start: 1992-02-01 00:00:00
End: 2026-08-01 00:00:00


### 輸出解讀

目前 New Orders：

- Shape：`(415, 2)` → 415 個月份、2 個欄位。
- `date`：datetime 日期型態。
- `new_orders`：整數數值型態。
- 日期空值：0。
- New Orders 空值：0。
- 日期範圍：1992-02 到 2026-08。

**白話來說：**這份資料不只是「看起來正常」，Python 也真的把日期當日期、訂單當數字，而且目前沒有空值。

因此 New Orders 的基本格式驗證通過，可以用同一個方法正式下載其他四份資料。


## Step 7｜正式下載全部 5 個 Series

### 這一格要做什麼？

使用前面建立的 `SERIES_CONFIG` 與下載函數，一次下載：

1. New Orders
2. Shipments
3. Unfilled Orders
4. Inventories
5. Industrial Production

每份資料會放進 `series_data` dictionary，並印出：

- Series ID
- 筆數
- 最早日期
- 最新日期

### 這一步主要要檢查什麼？

不同 Series 的起始年月不一定完全相同，因此先確認每份資料實際涵蓋的時間範圍，後面合併時才能知道為什麼某些月份會被保留或排除。


In [7]:
#正式下載全部五個 Series
series_data = {}

for column_name, metadata in SERIES_CONFIG.items():
    series_id = metadata["series_id"]

    dataframe = download_fred_series(
        series_id=series_id,
        column_name=column_name,
    )

    series_data[column_name] = dataframe

    print(
        f"{column_name:<22} "
        f"{series_id:<10} "
        f"{len(dataframe):>5} rows "
        f"{dataframe['date'].min().date()} "
        f"to {dataframe['date'].max().date()}"
    )

new_orders             A34HNO       415 rows 1992-02-01 to 2026-08-01
shipments              A34HVS       416 rows 1992-01-01 to 2026-08-01
unfilled_orders        A34HUO       416 rows 1992-01-01 to 2026-08-01
inventories            A34HTI       416 rows 1992-01-01 to 2026-08-01
industrial_production  IPG3344S     656 rows 1972-01-01 to 2026-08-01


### 輸出解讀

五個 Series 都成功下載：

- New Orders：415 rows，1992-02～2026-08
- Shipments：416 rows，1992-01～2026-08
- Unfilled Orders：416 rows，1992-01～2026-08
- Inventories：416 rows，1992-01～2026-08
- Industrial Production：656 rows，1972-01～2026-08

這裡最重要的發現是：**五份資料的起始月份不同。**

New Orders 少了最前面的 1992-01；Industrial Production 則早很多，從 1972 年就開始。

**白話來說：**這不是錯誤，而是不同官方統計系列本來就有不同的歷史長度。之後若要把五個欄位放在同一張表，就只能從它們能共同對上的期間開始。

**下一步：**先把下載到的原始資料保存起來。


## Step 8｜保存原始下載資料

### 這一格要做什麼？

把剛才從 FRED 下載的五份資料，各自存成 CSV 到：

`data/raw/`

### 為什麼要保存 Raw Data？

之後做分析時可以直接使用本地資料，不必每次重新連網下載。也能保留「當時實際使用的資料版本」，方便日後追蹤官方資料 revision。


In [8]:
#把原始資料各自存下來
for column_name, dataframe in series_data.items():
    output_path = RAW_DATA_DIR / f"{column_name}.csv"

    dataframe.to_csv(
        output_path,
        index=False,
    )

    print("Saved:", output_path.name)

Saved: new_orders.csv
Saved: shipments.csv
Saved: unfilled_orders.csv
Saved: inventories.csv
Saved: industrial_production.csv


### 輸出解讀

畫面顯示五份 CSV 都成功保存：

- `new_orders.csv`
- `shipments.csv`
- `unfilled_orders.csv`
- `inventories.csv`
- `industrial_production.csv`

**白話來說：**目前已經有一份「原始資料快照」存在 `data/raw/`，後面即使再做清理或計算，也不需要直接改動這些原始檔案。

**下一步：**開始正式檢查每份 Series 的重複、空日期與空數值。


## Step 9｜建立第一層 Data Validation Summary

### 這一格要做什麼？

針對每個 Series 統一檢查：

- `row_count`：總筆數。
- `date_start / date_end`：資料涵蓋期間。
- `duplicate_dates`：同一月份是否重複。
- `missing_dates`：日期本身是否解析失敗。
- `missing_values`：數值欄位是否有空值。

最後把五個 Series 的結果整理成同一張 `validation_summary`。

### 為什麼這比單純 `df.info()` 更好？

因為這張表可以直接保存成驗證報告，也可以讓不同資料系列用相同標準一起比較。


In [9]:
#開始做 Data Validation
validation_records = []

for column_name, dataframe in series_data.items():

    duplicate_date_count = int(
        dataframe["date"].duplicated().sum()
    )

    missing_date_count = int(
        dataframe["date"].isna().sum()
    )

    missing_value_count = int(
        dataframe[column_name].isna().sum()
    )

    validation_records.append(
        {
            "series": column_name,
            "series_id": SERIES_CONFIG[column_name]["series_id"],
            "row_count": len(dataframe),
            "date_start": dataframe["date"].min(),
            "date_end": dataframe["date"].max(),
            "duplicate_dates": duplicate_date_count,
            "missing_dates": missing_date_count,
            "missing_values": missing_value_count,
        }
    )

validation_summary = pd.DataFrame(validation_records)

validation_summary

,series,series_id,row_count,date_start,date_end,duplicate_dates,missing_dates,missing_values
0,new_orders,A34HNO,415,1992-02-01,2026-08-01,0,0,0
1,shipments,A34HVS,416,1992-01-01,2026-08-01,0,0,0
2,unfilled_orders,A34HUO,416,1992-01-01,2026-08-01,0,0,0
3,inventories,A34HTI,416,1992-01-01,2026-08-01,0,0,0
4,industrial_production,IPG3344S,656,1972-01-01,2026-08-01,0,0,0


### 輸出解讀

五個 Series 的驗證表顯示：

- `duplicate_dates = 0`
- `missing_dates = 0`
- `missing_values = 0`

全部都是 0。

這代表目前沒有：

- 同一月份被重複記錄。
- 日期解析失敗。
- 某個已存在月份的值是空白。

另外資料期間也和前一格一致。

**白話來說：**五份資料「每一列本身」都是乾淨的。

但這還不能證明月份完全連續，因為某個月份如果整列都消失，`missing_values` 仍然可能是 0。

**下一步：**專門檢查「月份整列不見」的情況。


## Step 10｜檢查「整個月份不見」的問題

### 這一格要做什麼？

一般的 `isna()` 只能找到「有這一列，但是值是空的」。

但如果資料直接從：

`2025-01 → 2025-02 → 2025-04`

那 2025-03 整列根本不存在，`isna()` 看不到。

因此 `find_missing_months()` 會：

1. 依資料的起訖日期建立完整的每月時間軸。
2. 跟實際出現的月份比較。
3. 找出中間是否有月份整個缺失。

接著對五個 Series 全部執行一次。

### 注意

這個檢查只檢查「各 Series 自己的有效日期範圍內」有沒有斷月，不會要求不同 Series 一定從同一個月份開始。


In [10]:
#檢查月份有沒有偷偷斷掉
def find_missing_months(dataframe: pd.DataFrame) -> pd.DatetimeIndex:

    valid_dates = (
        dataframe["date"]
        .dropna()
        .sort_values()
    )

    expected_dates = pd.date_range(
        start=valid_dates.min(),
        end=valid_dates.max(),
        freq="MS",
    )

    observed_dates = pd.DatetimeIndex(
        valid_dates.unique()
    )

    return expected_dates.difference(observed_dates)

#五份資料全部查缺月
missing_month_results = {}

for column_name, dataframe in series_data.items():

    missing_months = find_missing_months(dataframe)

    missing_month_results[column_name] = missing_months

    print(
        f"{column_name:<22} "
        f"missing months = {len(missing_months)}"
    )

    if len(missing_months) > 0:
        print(missing_months.tolist())

new_orders             missing months = 0
shipments              missing months = 0
unfilled_orders        missing months = 0
inventories            missing months = 0
industrial_production  missing months = 0


### 輸出解讀

結果顯示五個 Series：

`missing months = 0`

代表從每個 Series **自己的第一個有效月份到最後一個有效月份之間**，月份都是連續的，沒有中途跳過某一個月。

**特別注意：**

New Orders 從 1992-02 開始，但這裡仍然顯示 0 是正確的。因為 1992-01 在 New Orders 的官方 Series 中不是它有效日期範圍的一部分，所以不是「中間漏掉的月份」。

**白話來說：**目前每份時間序列都沒有斷掉，可以安全地按月份合併。

**下一步：**把五個 Series 合成一張 Planner 主資料表。


## Step 11｜合併成 Planner 主資料表

### 這一格要做什麼？

把五個 Series 依照 `date` 合併成同一張月度 Planner Dataset：

`date + new_orders + shipments + unfilled_orders + inventories + industrial_production`

主表從 `new_orders` 開始，再用 `left join` 加入其他 Series。

### 為什麼以 New Orders 為主？

這個專案後面最主要的 FCST 目標就是 New Orders，因此以它的有效月份作為主時間軸。其他指標只保留能和 New Orders 同月對上的資料。

### 重要限制

New Orders 從 1992-02 才開始，所以即使 Shipments、Backlog、Inventory 有 1992-01，合併後的 Planner 主表仍會從 1992-02 開始。這是資料可用期間不同造成的，不是資料遺失。


In [11]:
#合併成 Planner 主資料表
planning_data = series_data["new_orders"].copy()

for column_name in [
    "shipments",
    "unfilled_orders",
    "inventories",
    "industrial_production",
]:
    planning_data = planning_data.merge(
        series_data[column_name],
        on="date",
        how="left",
        validate="one_to_one",
    )

planning_data = planning_data.sort_values(
    "date"
).reset_index(drop=True)

planning_data.head()

,date,new_orders,shipments,unfilled_orders,inventories,industrial_production
0,1992-02-01,3411,3484,8656,7714,0.3629
1,1992-03-01,3711,3754,8613,7661,0.3675
2,1992-04-01,3745,3652,8706,7520,0.3725
3,1992-05-01,3735,3673,8768,7427,0.3777
4,1992-06-01,3814,3701,8881,7585,0.3903


### 輸出解讀

合併後每個月份現在同時有：

- New Orders
- Shipments
- Unfilled Orders
- Inventories
- Industrial Production

第一列是 **1992-02**，不是 1992-01，原因是我們以 `new_orders` 作為主時間軸，而 New Orders 本身從 1992-02 才有資料。

第一列例如：

- New Orders = 3,411
- Shipments = 3,484
- Unfilled Orders = 8,656
- Inventories = 7,714

這四個 M3 數值的單位都是百萬美元。

`industrial_production = 0.3629` 則是生產指數，不是 0.3629 百萬美元、也不是 0.3629%。它和前四個欄位的單位完全不同。

**白話來說：**我們已經把「需求進來多少、出貨多少、還欠多少訂單、庫存多少、生產活動如何」放在同一個月份上。

這張表就是後續 Planner 分析的核心資料。


## Step 12｜驗證合併後的主資料表

### 這一格要做什麼？

合併資料後重新檢查：

- 主表大小 `Shape`
- 合併後日期範圍
- 每個欄位的 Missing Values
- 日期是否重複

### 為什麼合併前檢查過，這裡還要再檢查？

因為每份資料單獨正常，不代表合併後一定正常。不同日期範圍或 join 方式可能造成新的空值，所以合併後必須再驗一次。


In [12]:
#確認合併後有沒有問題
print("Shape:", planning_data.shape)

print("\nDate range:")
print(
    planning_data["date"].min(),
    "to",
    planning_data["date"].max(),
)

print("\nMissing values:")
print(planning_data.isna().sum())

print("\nDuplicate dates:")
print(planning_data["date"].duplicated().sum())

Shape: (415, 6)

Date range:
1992-02-01 00:00:00 to 2026-08-01 00:00:00

Missing values:
date                     0
new_orders               0
shipments                0
unfilled_orders          0
inventories              0
industrial_production    0
dtype: int64

Duplicate dates:
0


### 輸出解讀

合併後主資料表：

- Shape：`(415, 6)`
- 日期：1992-02～2026-08
- 六個欄位 Missing Values 全部為 0
- Duplicate dates = 0

**白話來說：**415 個月份全部都能同時找到五個分析指標，合併沒有產生任何新的空值或重複月份。

這一步很重要，因為如果某個 Series 某月對不上，這裡通常就會出現 NaN。

**結論：**合併後資料結構正常。

**下一步：**把最重要的品質條件寫成自動 PASS / FAIL。


## Step 13｜建立自動 PASS / FAIL 規則

### 這一格要做什麼？

把最重要的資料品質條件變成可以自動判斷的布林規則：

- 六個必要欄位都有存在。
- 日期沒有重複。
- 日期沒有無效值。
- New Orders、Shipments、Unfilled Orders、Inventories 都沒有負數。

每個條件最後都會得到 `True` 或 `False`。

### 為什麼要自動化？

之後即使官方資料更新，也不需要每次人工重新看整張表。只要重新執行 Notebook，就能快速知道核心資料品質是否仍然符合要求。

> 補充：`industrial_production` 是指數，不是金額。本格目前沒有另外寫它的非負值檢查；如果之後要把驗證規則做得更完整，可以再加入。


In [13]:
#再加一個自動 PASS / FAIL
required_columns = [
    "date",
    "new_orders",
    "shipments",
    "unfilled_orders",
    "inventories",
    "industrial_production",
]

checks = {
    "required_columns_present": set(required_columns).issubset(
        planning_data.columns
    ),
    "no_duplicate_dates": (
        planning_data["date"].duplicated().sum() == 0
    ),
    "dates_are_valid": (
        planning_data["date"].notna().all()
    ),
    "new_orders_non_negative": (
        planning_data["new_orders"].dropna().ge(0).all()
    ),
    "shipments_non_negative": (
        planning_data["shipments"].dropna().ge(0).all()
    ),
    "unfilled_orders_non_negative": (
        planning_data["unfilled_orders"].dropna().ge(0).all()
    ),
    "inventories_non_negative": (
        planning_data["inventories"].dropna().ge(0).all()
    ),
}

validation_checks = pd.DataFrame(
    {
        "check": checks.keys(),
        "passed": checks.values(),
    }
)

validation_checks

,check,passed
0,required_columns_present,True
1,no_duplicate_dates,True
2,dates_are_valid,True
3,new_orders_non_negative,True
4,shipments_non_negative,True
5,unfilled_orders_non_negative,True
6,inventories_non_negative,True


### 輸出解讀

七項核心檢查全部是 `True`：

- 必要欄位都有。
- 日期沒有重複。
- 日期都有效。
- New Orders 沒有負數。
- Shipments 沒有負數。
- Unfilled Orders 沒有負數。
- Inventories 沒有負數。

**白話來說：**目前這份 Planner Dataset 通過我們設定的核心資料品質門檻。

這些非負檢查也符合商業常識：訂單金額、出貨價值、backlog 與 inventory 不應出現負值。

**嚴格補充：**目前程式沒有另外檢查 `industrial_production` 是否非負，也沒有檢查五個 Series 的單位 metadata 是否被官方修改。現階段不影響這次 PASS，但如果後面要把專案做成更完整的 production-style pipeline，可以再補。


## Step 14｜只有全部驗證通過才保存 Processed Data

### 這一格要做什麼？

這一步建立一道「資料品質閘門」。

只有當上一格所有 `passed == True` 時，才會把主資料表存成：

`data/processed/monthly_planning_data.csv`

如果有任何條件失敗，就不產生 processed dataset，而是印出失敗項目。

### 為什麼這個設計重要？

它可以避免後面 EDA、Forecast 或 Planner KPI 不小心使用到明知有問題的資料。


In [14]:
#只在全部 PASS 時存 processed data
if validation_checks["passed"].all():

    output_path = (
        PROCESSED_DATA_DIR
        / "monthly_planning_data.csv"
    )

    planning_data.to_csv(
        output_path,
        index=False,
    )

    print("VALIDATION PASSED")
    print("Saved:", output_path)

else:
    print("VALIDATION FAILED")
    print(
        validation_checks.loc[
            ~validation_checks["passed"]
        ]
    )

VALIDATION PASSED
Saved: C:\Users\momo8\Desktop\electronic-components-planning\data\processed\monthly_planning_data.csv


### 輸出解讀

輸出：

`VALIDATION PASSED`

代表上一格所有核心品質規則都通過，因此程式才真正產生：

`data/processed/monthly_planning_data.csv`

**白話來說：**Raw Data 已經通過驗證，現在才正式升級成「後續可以拿來分析的 Processed Data」。

如果前面的任何檢查失敗，這一格就不會默默保存有問題的資料。

**下一步：**把驗證證據另外保存成報告。


## Step 15｜保存資料驗證報告

### 這一格要做什麼？

最後把兩張資料品質報告保存到 `reports/tables/`：

- `data_validation_summary.csv`：每個 Series 的筆數、日期範圍、重複與缺值情況。
- `data_validation_checks.csv`：核心 PASS / FAIL 檢查。

### 為什麼除了資料本身還要留報告？

Processed Data 告訴我們「可以分析什麼」，Validation Report 則記錄「為什麼我們相信這份資料可以使用」。對作品集來說，這能清楚呈現資料品質與可追溯性。


In [15]:
#把驗證報告也存起來
validation_summary.to_csv(
    REPORT_TABLE_DIR / "data_validation_summary.csv",
    index=False,
)

validation_checks.to_csv(
    REPORT_TABLE_DIR / "data_validation_checks.csv",
    index=False,
)

print("Validation reports saved.")

Validation reports saved.


### 輸出解讀

輸出：

`Validation reports saved.`

代表兩份驗證報告已成功保存：

1. `data_validation_summary.csv`
2. `data_validation_checks.csv`

**白話來說：**這本 Notebook 不只是產生一份乾淨資料，也留下了「資料為什麼被判定可用」的紀錄。

## 本 Notebook 最終結論

目前資料驗證結果：

- 五個官方 Series 皆成功下載。
- 各 Series 沒有重複日期。
- 沒有日期解析失敗。
- 沒有資料空值。
- 各自有效期間內沒有缺月。
- 合併後 415 個月、6 個欄位完整對齊。
- 核心 PASS / FAIL 全部通過。
- Processed Dataset 已成功建立。

因此可以進入下一階段：

**`02_demand_shipment_analysis.ipynb`**

下一本會開始回答 Planner 真正關心的問題：New Orders 和 Shipments 的變化是否一致、訂單進來的速度是否超過出貨速度，以及 Backlog 是否開始累積。
